# MedGemma 4B-PT — Inférence JSON sans fine-tuning (RSNA)

**Objectif :** générer une classification structurée en JSON à partir d'une radiographie
thoracique, en utilisant le modèle de base `google/medgemma-4b-pt` **sans entraînement,
sans adaptateur LoRA**. Le JSON est obtenu uniquement grâce au prompt.


/!\ Prototype pédagogique. Non destiné au diagnostic. Un clinicien qualifié doit vérifier l'image.

---

## Pipeline

```
Image CXR (PNG)
→ prompt système + image
→ MedGemma-4b-pt
→ texte généré
→ extraction + validation JSON
→ garde-fous (uncertain si invalide)
```

Le modèle n'est pas entraîné : on l'utilise en génération directe (zero-shot) avec un
prompt qui impose le format de sortie.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 0. Variables d'environnement & reproductibilité
# ═══════════════════════════════════════════════════════════════════
# À exécuter EN PREMIER, avant tout import PyTorch/CUDA.

import os, random

SEED = 42

def set_all_seeds(seed: int = SEED) -> None:
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    try:
        import numpy as np
        np.random.seed(seed)
    except ImportError:
        pass
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

set_all_seeds(SEED)

# Réduit la fragmentation mémoire GPU (sans effet si CPU)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

print(f'Seed fixée à {SEED}')

In [ ]:
# ════════════════════════════════════
# STEP 1. Installation des dépendances
# ════════════════════════════════════
# À exécuter une seule fois dans l'environnement JupyterLab local.
# transformers + accelerate pour le modèle, bitsandbytes UNIQUEMENT si GPU CUDA.

# Décommentez la ligne adaptée à votre machine :

# --- GPU NVIDIA (CUDA) : chargement 4-bit possible ---
# %pip install -q "transformers>=4.51.3" accelerate bitsandbytes pillow

# --- CPU uniquement (pas de GPU NVIDIA) : sans bitsandbytes ---
# %pip install -q "transformers>=4.51.3" accelerate pillow

print("Décommentez la ligne d'installation adaptée à votre machine, puis ré-exécutez.")

In [ ]:
# ═══════════════════════════════════════════════
# STEP 2. Configuration : chemins et périphérique
# ═══════════════════════════════════════════════
# Adaptez IMAGES_DIR au dossier contenant vos radiographies PNG en local.

from pathlib import Path
import torch

MODEL_ID = 'google/medgemma-4b-pt'

# Dossier local contenant les images à classifier
IMAGES_DIR = Path('data/RSNA/Test/Images')

# Détection automatique du périphérique
if torch.cuda.is_available():
    DEVICE = 'cuda'
    USE_4BIT = True   # quantification 4-bit sur GPU pour réduire la VRAM
    print(f'GPU détecté : {torch.cuda.get_device_name(0)}')
else:
    DEVICE = 'cpu'
    USE_4BIT = False  # bitsandbytes n'est pas utilisable sur CPU
    print('Aucun GPU CUDA — exécution sur CPU (plus lent, sans quantification 4-bit).')

print(f'Périphérique : {DEVICE} | Quantification 4-bit : {USE_4BIT}')

In [ ]:
# ══════════════════════════════════════════
# STEP 3. Authentification HuggingFace Hub
# ══════════════════════════════════════════
# MedGemma est un modèle "gated" : il faut un token HF et avoir accepté
# les conditions sur https://huggingface.co/google/medgemma-4b-pt

from huggingface_hub import login

# Renseignez votre token (https://huggingface.co/settings/tokens)
# login(token='hf_...')

# Ou, si déjà connecté via `huggingface-cli login` en terminal, cette cellule
# peut être laissée telle quelle.
print('Si le chargement du modèle échoue en 401/403, exécutez login(token="hf_...").')

In [ ]:
# ══════════════════════════════════════════════════
# STEP 4. Chargement de MedGemma 4B-PT (inférence)
# ══════════════════════════════════════════════════
# Pas de LoRA, pas de prepare_model_for_kbit_training : inférence pure.

import warnings
from transformers import AutoProcessor, AutoModelForImageTextToText

warnings.filterwarnings('ignore', category=FutureWarning)

processor = AutoProcessor.from_pretrained(MODEL_ID)

if USE_4BIT:
    # GPU : quantification 4-bit NF4 pour réduire la VRAM
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={'': 0},
        dtype=torch.bfloat16,
    )
else:
    # CPU : pas de quantification, float32 pour compatibilité maximale
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,
        dtype=torch.float32,
        low_cpu_mem_usage=True,
    )
    model.to(DEVICE)

model.eval()  # mode inférence : désactive dropout, etc.
print('Modèle chargé en mode inférence.')
print('Device:', next(model.parameters()).device)

In [ ]:
# ═══════════════════════════════════════════
# STEP 5. Prompt de classification (zero-shot)
# ═══════════════════════════════════════════
# Le JSON est obtenu grâce à ce prompt, PAS grâce à un entraînement.

BOI = processor.boi_token  # token de début d'image

SYSTEM_PROMPT = (
    'You are an educational radiology assistant for engineering students. '
    'You are not a clinician. Analyze the frontal chest X-ray and return '
    'only the classification as valid JSON.'
)

USER_PROMPT = (
    'Classify this frontal chest X-ray. '
    'Return ONLY valid JSON with exactly these fields:\n'
    "{\n"
    '  "image_quality": "good | limited | poor",\n'
    '  "predicted_class": "normal | suspected_opacity | uncertain",\n'
    '  "confidence": 0.0,\n'
    '  "visual_evidence": ["short factual observation"],\n'
    '  "justification": "2 to 4 cautious sentences based only on visible evidence",\n'
    '  "limitations": ["image quality", "projection", "lack of clinical context"],\n'
    '  "warning": "Educational prototype only. Not for diagnosis. A qualified clinician must verify the image."\n'
    "}\n"
    'Rules:\n'
    '- Use "suspected_opacity" when there is a visible pulmonary opacity or consolidation.\n'
    '- Use "normal" when no significant abnormality is visible.\n'
    '- Use "uncertain" when image quality is poor or the findings are ambiguous.\n'
    '- Return ONLY the JSON object, no extra text before or after.'
)

def build_prompt() -> str:
    """Prompt multi-tour Gemma, SANS réponse model (le modèle doit la générer)."""
    return (
        f'<start_of_turn>system\n{SYSTEM_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>user\n{BOI}\n{USER_PROMPT}<end_of_turn>\n'
        f'<start_of_turn>model\n'  # <-- s'arrête ici : le modèle complète
    )

print('Prompt prêt.')
print(build_prompt()[:300], '...')

In [ ]:
# ═══════════════════════════════════════════════════════
# STEP 6. Garde-fous : extraction et validation du JSON
# ═══════════════════════════════════════════════════════
# Le modèle de base (non fine-tuné) peut produire du texte autour du JSON,
# ou un JSON incomplet. Ces garde-fous garantissent une sortie conforme.

import json
import re

REQUIRED_KEYS = [
    'image_quality', 'predicted_class', 'confidence',
    'visual_evidence', 'justification', 'limitations', 'warning',
]
VALID_CLASSES = {'normal', 'suspected_opacity', 'uncertain'}
VALID_QUALITY = {'good', 'limited', 'poor'}

WARNING_TEXT = (
    'Educational prototype only. Not for diagnosis. '
    'A qualified clinician must verify the image.'
)

def fallback_uncertain(reason: str) -> dict:
    """Sortie conservatrice conforme au schéma quand le JSON est inexploitable."""
    return {
        'image_quality': 'poor',
        'predicted_class': 'uncertain',
        'confidence': 0.0,
        'visual_evidence': [],
        'justification': f'Model output could not be parsed as valid JSON ({reason}). '
                         'Returning uncertain as a safe default.',
        'limitations': ['unparseable model output', 'no clinical context'],
        'warning': WARNING_TEXT,
    }

def extract_json(text: str) -> dict:
    """Extrait le premier objet JSON du texte généré et le valide/complète."""
    # Cherche le premier bloc {...} dans la sortie
    match = re.search(r'\{.*\}', text, re.DOTALL)
    if not match:
        return fallback_uncertain('no JSON object found')
    try:
        data = json.loads(match.group(0))
    except json.JSONDecodeError as e:
        return fallback_uncertain(f'JSONDecodeError: {e}')

    # Complétion des clés manquantes + validation des valeurs
    if data.get('predicted_class') not in VALID_CLASSES:
        data['predicted_class'] = 'uncertain'
    if data.get('image_quality') not in VALID_QUALITY:
        data['image_quality'] = 'poor'
    try:
        data['confidence'] = float(data.get('confidence', 0.0))
    except (TypeError, ValueError):
        data['confidence'] = 0.0

    data.setdefault('visual_evidence', [])
    data.setdefault('justification', '')
    data.setdefault('limitations', [])
    # Le warning est toujours forcé (garde-fou non négociable)
    data['warning'] = WARNING_TEXT

    # Réordonne selon le schéma
    return {k: data.get(k) for k in REQUIRED_KEYS}

print('Garde-fous JSON prêts.')

In [ ]:
# ═══════════════════════════════════════════
# STEP 7. Fonction d'inférence sur une image
# ═══════════════════════════════════════════

from PIL import Image

def predict(image_path) -> dict:
    """Charge une image, interroge MedGemma, retourne un dict JSON validé."""
    image = Image.open(image_path).convert('RGB')
    prompt = build_prompt()

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors='pt',
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,   # déterministe
        )

    # Ne décode que les tokens générés (après le prompt)
    generated = output_ids[0][inputs['input_ids'].shape[1]:]
    text = processor.decode(generated, skip_special_tokens=True)

    return extract_json(text)

print('Fonction predict() prête.')

In [ ]:
# ═══════════════════════════════════════════
# STEP 8. Test sur une image
# ═══════════════════════════════════════════
# Adaptez le chemin vers une de vos images locales.

# Exemple : première image trouvée dans IMAGES_DIR
sample_images = sorted(IMAGES_DIR.glob('*.png')) if IMAGES_DIR.exists() else []

if sample_images:
    test_path = sample_images[0]
    print(f'Test sur : {test_path.name}\n')
    result = predict(test_path)
    print(json.dumps(result, indent=2, ensure_ascii=False))
else:
    print(f'Aucune image .png trouvée dans {IMAGES_DIR}.')
    print('Renseignez IMAGES_DIR (STEP 2) ou passez un chemin directement :')
    print("  result = predict('chemin/vers/image.png')")

In [ ]:
# ═══════════════════════════════════════════════════
# STEP 9. Inférence en lot + export CSV des résultats
# ═══════════════════════════════════════════════════
# Traite un ensemble d'images et enregistre les prédictions dans un CSV.

import csv

def predict_batch(image_paths, out_csv='medgemma_predictions.csv', limit=None):
    """Classifie une liste d'images et écrit les résultats dans un CSV."""
    paths = list(image_paths)
    if limit:
        paths = paths[:limit]

    rows = []
    for i, p in enumerate(paths, 1):
        try:
            pred = predict(p)
        except Exception as e:
            pred = fallback_uncertain(f'exception: {e}')
        rows.append({
            'case_id': Path(p).stem,
            'image_path': str(p),
            'predicted_class': pred['predicted_class'],
            'image_quality': pred['image_quality'],
            'confidence': pred['confidence'],
            'justification': pred['justification'],
        })
        print(f'[{i}/{len(paths)}] {Path(p).name} -> {pred["predicted_class"]}')

    with open(out_csv, 'w', newline='', encoding='utf-8') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print(f'\n{len(rows)} prédictions écrites dans {out_csv}')
    return rows

# Exemple : traite les 10 premières images (retirez limit pour tout traiter)
# results = predict_batch(sorted(IMAGES_DIR.glob('*.png')), limit=10)
print('Fonction predict_batch() prête.')
print('Décommentez la dernière ligne pour lancer un lot.')

## Notes techniques

**Modèle :** `google/medgemma-4b-pt` (base pré-entraîné, vision-language), utilisé
**sans fine-tuning ni adaptateur LoRA**. La classification structurée est obtenue
uniquement par le prompt (approche zero-shot).

**Pourquoi cette approche ?** Le fine-tuning QLoRA nécessitait un GPU NVIDIA avec
assez de VRAM, indisponible ici. L'inférence directe du modèle de base ne demande
pas d'entraînement — elle correspond à la voie P0 « Prompting » (rapide, aucune
optimisation matérielle lourde).

**Garde-fous :** le modèle de base n'est pas garanti de produire un JSON parfait.
`extract_json()` extrait le premier objet JSON, complète les clés manquantes,
valide les classes/qualités autorisées, force le champ `warning`, et retombe sur
`uncertain` si la sortie est inexploitable. C'est cohérent avec la logique de
`src/guardrails.py` du projet.

**Performance CPU :** sans GPU, chaque inférence prend plusieurs minutes. Pour un
test rapide, limitez le nombre d'images (`limit=` dans `predict_batch`).

**Limites :** un modèle de base non spécialisé produit des classifications de
qualité variable. Ces résultats sont des sorties expérimentales, jamais un
diagnostic. Voir README pour la position non clinique du projet.